# Two-State Benchmark

This notebook reads saved training outputs from `results/twostate`. It does not launch training.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from mfc.visualization import (
    best_runs_by_label,
    load_runs,
    objective_table,
    plot_flow_comparison,
    plot_state_flow,
    plot_validation_rewards,
    runtime_table,
    twostate_policy_error_table,
)

ENV = 'twostate'
RESULTS_ROOT = ROOT / 'results'
runs = load_runs(RESULTS_ROOT, env=ENV)
print(f'Loaded {len(runs)} saved runs from {RESULTS_ROOT / ENV}')

## Validation Reward

Mean validation reward over training, with one standard deviation across seeds. Exact-flow and particle-flow runs are shown separately so their seed averages are not mixed.

In [ ]:
if not runs:
    print('No saved runs yet. Run scripts/run.py before executing the analysis cells.')
else:
    for horizon in sorted({run['metadata']['horizon'] for run in runs}):
        for flow in sorted({run['metadata']['flow'] for run in runs if run['metadata']['horizon'] == horizon}):
            fig, ax = plt.subplots(figsize=(8, 4.5))
            try:
                plot_validation_rewards(runs, env=ENV, horizon=horizon, flow=flow, ax=ax)
                ax.set_title(f'Two-state validation reward, T={horizon}, flow={flow}')
                plt.show()
            except ValueError as exc:
                plt.close(fig)
                print(exc)

## Learned Policy Error

Average and maximum absolute errors between the learned action probabilities and the closed-form optimal stationary policy.

In [ ]:
policy_errors = twostate_policy_error_table(runs)
if policy_errors.empty:
    print('No two-state policy table available yet.')
else:
    display(policy_errors.groupby(['label', 'flow', 'horizon'], as_index=False).agg(
        mean_abs_policy_error=('mean_abs_policy_error', 'mean'),
        mean_abs_policy_error_std=('mean_abs_policy_error', 'std'),
        max_abs_policy_error=('max_abs_policy_error', 'mean'),
    ).fillna(0.0))

## Learned State Flows

State-probability trajectories induced by the best saved policy for each algorithm, perturbation, horizon, and flow setting.

In [ ]:
for run in best_runs_by_label(runs):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    plot_state_flow(run, ax=ax)
    meta = run['metadata']
    ax.set_title(f"{meta['algorithm']}, perturbation={meta['perturbation']}, T={meta['horizon']}, flow={meta['flow']}")
    plt.show()

## Exact vs Particle Flow

Transport validation curves split by flow estimator. This highlights how much the learned behavior changes when the population flow is estimated with particles.

In [ ]:
if runs:
    for horizon in sorted({run['metadata']['horizon'] for run in runs}):
        fig, ax = plt.subplots(figsize=(8, 4.5))
        try:
            plot_flow_comparison(runs, env=ENV, horizon=horizon, ax=ax)
            ax.set_title(f'Transport flow comparison, T={horizon}')
            plt.show()
        except ValueError as exc:
            plt.close(fig)
            print(exc)

## Objective and Runtime Tables

Summary tables for final validation reward, estimated simulator budget, and runtime. Exact objective columns appear only for environments exposing a closed-form oracle.

In [ ]:
display(objective_table(runs))
display(runtime_table(runs))